# 4-dars: Unsupervised Learning

## 📚 Dars maqsadi
Ushbu darsda biz unsupervised learning algoritmlarini o'rganamiz:
- **K-means Clustering** - Ma'lumotlarni k ta guruhga ajratish
- **Hierarchical Clustering** - Daraxtsimon klasterlash
- **PCA (Principal Component Analysis)** - Dimensionality reduction

---

In [ ]:
# Kerakli kutubxonalarni import qilish
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.datasets import make_blobs, load_iris, load_wine
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

# Vizualizatsiya sozlamalari
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Barcha kutubxonalar muvaffaqiyatli yuklandi!")

---

# 🎯 Supervised vs Unsupervised Learning

## Farq nima?

### Supervised Learning (Nazorat ostida o'rganish)
- **Input**: X (features) + Y (labels)
- **Maqsad**: X → Y mapping'ni o'rganish
- **Misollar**: Classification, Regression
- **Real life**: Imtihon - javoblari bor

### Unsupervised Learning (Nazorat​siz o'rganish)
- **Input**: Faqat X (features), Y yo'q!
- **Maqsad**: Pattern, structure topish
- **Misollar**: Clustering, Dimensionality Reduction
- **Real life**: Kutubxonada kitoblarni guruhlash - hech kim javob bermaydi

---

# 1️⃣ K-means Clustering

## 📖 Nazariya

**K-means** - eng mashhur clustering algoritmi. Ma'lumotlarni **K ta klasterga** ajratadi.

### Algoritm qadamlari:

1. **Initialize**: K ta random centroid tanlash
2. **Assignment**: Har bir nuqtani eng yaqin centroid'ga biriktirish
3. **Update**: Har bir klasterning yangi centroid'ini hisoblash (o'rtacha)
4. **Repeat**: 2-3 qadam convergence bo'lguncha takrorlanadi

### Matematik formula:

**Objective (minimize qilish kerak)**:
$$J = \sum_{k=1}^{K} \sum_{x_i \in C_k} ||x_i - \mu_k||^2$$

Bu yerda:
- $C_k$ - k-klaster
- $\mu_k$ - k-klasterning centroid'i
- $||x_i - \mu_k||^2$ - nuqtadan centroid'gacha masofa kvadrati

### Centroid yangilanishi:
$$\mu_k = \frac{1}{|C_k|} \sum_{x_i \in C_k} x_i$$

(Klasterdagi barcha nuqtalarning o'rtachasi)

---

## 🎨 K-means: Vizual Tushuntirish

Keling, K-means algoritmi qanday ishlashini qadam-baqadam ko'raylik.

In [ ]:
# Oddiy 2D data yaratish
np.random.seed(42)
X_demo, y_demo = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)

# K-means step-by-step visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

# Step 0: Original Data
axes[0].scatter(X_demo[:, 0], X_demo[:, 1], s=50, alpha=0.6, edgecolors='k')
axes[0].set_title('Qadam 0: Original Data\n(Label yo\'q!)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].grid(True, alpha=0.3)

# Step 1: Random Centroids
np.random.seed(10)
initial_centroids = X_demo[np.random.choice(X_demo.shape[0], 3, replace=False)]

axes[1].scatter(X_demo[:, 0], X_demo[:, 1], s=50, alpha=0.6, c='gray', edgecolors='k')
axes[1].scatter(initial_centroids[:, 0], initial_centroids[:, 1], 
                s=300, marker='X', c=['red', 'blue', 'green'], 
                edgecolors='black', linewidths=2, label='Centroids')
axes[1].set_title('Qadam 1: Random Centroids', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Steps 2-6: K-means iterations
for i, n_iter in enumerate([1, 2, 5, 10]):
    kmeans_iter = KMeans(n_clusters=3, init=initial_centroids, n_init=1, 
                         max_iter=n_iter, random_state=42)
    labels = kmeans_iter.fit_predict(X_demo)
    
    axes[i+2].scatter(X_demo[:, 0], X_demo[:, 1], s=50, 
                      c=labels, alpha=0.6, cmap='viridis', edgecolors='k')
    axes[i+2].scatter(kmeans_iter.cluster_centers_[:, 0], 
                      kmeans_iter.cluster_centers_[:, 1],
                      s=300, marker='X', c=['red', 'blue', 'green'],
                      edgecolors='black', linewidths=2)
    
    inertia = kmeans_iter.inertia_
    axes[i+2].set_title(f'Qadam {i+2}: Iteration {n_iter}\nInertia: {inertia:.2f}', 
                        fontsize=13, fontweight='bold')
    axes[i+2].set_xlabel('Feature 1')
    axes[i+2].set_ylabel('Feature 2')
    axes[i+2].grid(True, alpha=0.3)

plt.suptitle('K-means Algoritmi: Qadam-baqadam', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("K-MEANS ALGORITMI - TUSHUNTIRISH")
print("="*70)
print("\n📍 Qadam 0: Original data (label yo'q)")
print("📍 Qadam 1: 3 ta random centroid tanlanadi (X belgi)")
print("📍 Qadam 2-6: Har bir iteratsiyada:")
print("     1. Har bir nuqta eng yaqin centroid'ga biriktiriladi")
print("     2. Centroid'lar yangilanadi (klaster o'rtachasi)")
print("     3. Inertia (xatolik) kamayadi")
print("\n💡 Convergence: Centroid'lar harakatlanmasa, to'xtaydi!")
print("="*70)

## 💻 K-means: Amaliy Misol (Iris Dataset)

In [ ]:
# Iris datasetini yuklash
iris = load_iris()
X_iris = iris.data
y_iris_true = iris.target  # Clustering'da ishlatmaymiz, faqat compare uchun

# Faqat 2 ta feature (visualization uchun)
X_iris_2d = X_iris[:, [0, 2]]  # sepal length va petal length

# K-means (K=3)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
y_pred = kmeans.fit_predict(X_iris_2d)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Predicted clusters
axes[0].scatter(X_iris_2d[:, 0], X_iris_2d[:, 1], c=y_pred, 
                s=100, alpha=0.6, cmap='viridis', edgecolors='k', linewidth=1)
axes[0].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                s=400, marker='X', c='red', edgecolors='black', linewidths=3, 
                label='Centroids')
axes[0].set_xlabel('Sepal Length (cm)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Petal Length (cm)', fontsize=12, fontweight='bold')
axes[0].set_title('K-means Clustering (K=3)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# True labels (comparison)
axes[1].scatter(X_iris_2d[:, 0], X_iris_2d[:, 1], c=y_iris_true,
                s=100, alpha=0.6, cmap='viridis', edgecolors='k', linewidth=1)
axes[1].set_xlabel('Sepal Length (cm)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Petal Length (cm)', fontsize=12, fontweight='bold')
axes[1].set_title('True Labels (Taqqoslash uchun)', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Metrics
inertia = kmeans.inertia_
silhouette = silhouette_score(X_iris_2d, y_pred)

print("\n" + "="*60)
print("K-MEANS NATIJALAR (Iris Dataset)")
print("="*60)
print(f"Inertia (WCSS): {inertia:.2f}")
print(f"Silhouette Score: {silhouette:.4f}")
print(f"\nCentroids:")
print(kmeans.cluster_centers_)
print("\n📊 Cluster sizes:")
unique, counts = np.unique(y_pred, return_counts=True)
for cluster, count in zip(unique, counts):
    print(f"  Cluster {cluster}: {count} samples")
print("="*60)

## 🔍 Optimal K ni topish: Elbow Method

**Muammo**: K (klasterlar soni) ni qanday tanlash kerak?

**Yechim**: **Elbow Method** - turli K qiymatlari uchun **Inertia** ni hisoblash.

**Inertia (Within-Cluster Sum of Squares - WCSS)**:
$$WCSS = \sum_{k=1}^{K} \sum_{x_i \in C_k} ||x_i - \mu_k||^2$$

- Kichik Inertia = yaxshi clustering
- K ko'paysa, Inertia kamayadi
- Lekin K juda katta bo'lsa, overfitting!

In [ ]:
# Elbow Method
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    kmeans_elbow = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_elbow.fit(X_iris_2d)
    inertias.append(kmeans_elbow.inertia_)
    silhouettes.append(silhouette_score(X_iris_2d, kmeans_elbow.labels_))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Elbow Method - Inertia
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=10)
axes[0].set_xlabel('K (Number of Clusters)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Inertia (WCSS)', fontsize=12, fontweight='bold')
axes[0].set_title('Elbow Method - Inertia', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=3, color='red', linestyle='--', linewidth=2, label='Optimal K=3')
axes[0].legend(fontsize=11)

# Silhouette Score
axes[1].plot(K_range, silhouettes, 'ro-', linewidth=2, markersize=10)
axes[1].set_xlabel('K (Number of Clusters)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Silhouette Score', fontsize=12, fontweight='bold')
axes[1].set_title('Silhouette Score vs K', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
optimal_k = K_range[np.argmax(silhouettes)]
axes[1].axvline(x=optimal_k, color='red', linestyle='--', linewidth=2, 
                label=f'Best K={optimal_k}')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("OPTIMAL K NI TOPISH")
print("="*70)
print("\n📊 Elbow Method:")
print("  1. Turli K qiymatlari uchun Inertia hisoblash")
print("  2. Inertia vs K grafigini chizish")
print("  3. 'Elbow' (tirsak) nuqtani topish")
print("\n💡 'Elbow' - Inertia kamayishi sekinlashadigan nuqta")
print(f"\n🎯 Ushbu misolda optimal K = 3")
print(f"   - Eng yuqori Silhouette Score: {max(silhouettes):.4f} (K={optimal_k})")
print("="*70)

## 📊 Silhouette Analysis

**Silhouette Score** - har bir nuqta qanchalik yaxshi klasterlanganligi.

**Formula**:
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

Bu yerda:
- $a(i)$ - o'z klasteridagi boshqa nuqtalarga o'rtacha masofa
- $b(i)$ - eng yaqin boshqa klasterdagi nuqtalarga o'rtacha masofa

**Qiymatlar**:
- **1**: Perfect clustering
- **0**: Klaster chegarasida
- **-1**: Noto'g'ri klasterga biriktirilgan

In [ ]:
# Silhouette Analysis for K=2, 3, 4
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, k in enumerate([2, 3, 4]):
    kmeans_sil = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_sil = kmeans_sil.fit_predict(X_iris_2d)
    
    silhouette_vals = silhouette_samples(X_iris_2d, labels_sil)
    silhouette_avg = silhouette_score(X_iris_2d, labels_sil)
    
    y_lower = 10
    for i in range(k):
        cluster_silhouette_vals = silhouette_vals[labels_sil == i]
        cluster_silhouette_vals.sort()
        
        size_cluster_i = cluster_silhouette_vals.shape[0]
        y_upper = y_lower + size_cluster_i
        
        color = plt.cm.viridis(float(i) / k)
        axes[idx].fill_betweenx(np.arange(y_lower, y_upper),
                                 0, cluster_silhouette_vals,
                                 facecolor=color, edgecolor=color, alpha=0.7)
        
        axes[idx].text(-0.05, y_lower + 0.5 * size_cluster_i, str(i), 
                       fontsize=12, fontweight='bold')
        y_lower = y_upper + 10
    
    axes[idx].set_xlabel('Silhouette Coefficient', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Cluster', fontsize=11, fontweight='bold')
    axes[idx].set_title(f'K={k}\nAvg Score: {silhouette_avg:.3f}', 
                        fontsize=13, fontweight='bold')
    axes[idx].axvline(x=silhouette_avg, color='red', linestyle='--', linewidth=2)
    axes[idx].set_xlim([-0.1, 1])
    axes[idx].grid(True, alpha=0.3, axis='x')

plt.suptitle('Silhouette Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Silhouette Plot Tushuntirish:")
print("  - Har bir klaster alohida rang bilan ko'rsatilgan")
print("  - Qizil chiziq: O'rtacha silhouette score")
print("  - Keng va yuqori plot'lar: Yaxshi klasterlangan")
print("  - Tor yoki salbiy qiymatlar: Yomon klasterlangan")

### ⚠️ K-means Limitations (Cheklovlar)

K-means har doim ham yaxshi ishlamaydi:

1. **K ni tanlash qiyin** - domain knowledge kerak
2. **Spherical clusters** - faqat dumaloq klasterlar
3. **Equal size clusters** - har xil o'lchamdagi klasterlar bilan yomon ishlaydi
4. **Sensitive to outliers** - outlier'lar natijani buzadi
5. **Random initialization** - har safar turli natija (n_init ko'p bo'lishi kerak)

Keling, buni ko'rsatamiz:

In [ ]:
# K-means Limitations Demo
from sklearn.datasets import make_moons, make_circles

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Dataset 1: Well-separated blobs (yaxshi ishlaydi)
X1, y1 = make_blobs(n_samples=300, centers=3, cluster_std=0.6, random_state=42)
kmeans1 = KMeans(n_clusters=3, random_state=42)
labels1 = kmeans1.fit_predict(X1)

axes[0, 0].scatter(X1[:, 0], X1[:, 1], c=y1, cmap='viridis', s=50, alpha=0.6, edgecolors='k')
axes[0, 0].set_title('Original: Well-separated', fontsize=12, fontweight='bold')
axes[1, 0].scatter(X1[:, 0], X1[:, 1], c=labels1, cmap='viridis', s=50, alpha=0.6, edgecolors='k')
axes[1, 0].scatter(kmeans1.cluster_centers_[:, 0], kmeans1.cluster_centers_[:, 1],
                   s=300, marker='X', c='red', edgecolors='black', linewidths=2)
axes[1, 0].set_title('K-means: ✅ Yaxshi!', fontsize=12, fontweight='bold', color='green')

# Dataset 2: Moons (non-spherical)
X2, y2 = make_moons(n_samples=300, noise=0.1, random_state=42)
kmeans2 = KMeans(n_clusters=2, random_state=42)
labels2 = kmeans2.fit_predict(X2)

axes[0, 1].scatter(X2[:, 0], X2[:, 1], c=y2, cmap='viridis', s=50, alpha=0.6, edgecolors='k')
axes[0, 1].set_title('Original: Non-spherical (Moons)', fontsize=12, fontweight='bold')
axes[1, 1].scatter(X2[:, 0], X2[:, 1], c=labels2, cmap='viridis', s=50, alpha=0.6, edgecolors='k')
axes[1, 1].scatter(kmeans2.cluster_centers_[:, 0], kmeans2.cluster_centers_[:, 1],
                   s=300, marker='X', c='red', edgecolors='black', linewidths=2)
axes[1, 1].set_title('K-means: ❌ Yomon!', fontsize=12, fontweight='bold', color='red')

# Dataset 3: Circles (nested clusters)
X3, y3 = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)
kmeans3 = KMeans(n_clusters=2, random_state=42)
labels3 = kmeans3.fit_predict(X3)

axes[0, 2].scatter(X3[:, 0], X3[:, 1], c=y3, cmap='viridis', s=50, alpha=0.6, edgecolors='k')
axes[0, 2].set_title('Original: Nested Circles', fontsize=12, fontweight='bold')
axes[1, 2].scatter(X3[:, 0], X3[:, 1], c=labels3, cmap='viridis', s=50, alpha=0.6, edgecolors='k')
axes[1, 2].scatter(kmeans3.cluster_centers_[:, 0], kmeans3.cluster_centers_[:, 1],
                   s=300, marker='X', c='red', edgecolors='black', linewidths=2)
axes[1, 2].set_title('K-means: ❌ Juda yomon!', fontsize=12, fontweight='bold', color='red')

for ax in axes.flat:
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.grid(True, alpha=0.3)

plt.suptitle('K-means: Qachon yaxshi va qachon yomon ishlaydi?', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("K-MEANS CHEKLOVLARI")
print("="*70)
print("\n✅ Yaxshi ishlaydi:")
print("  - Spherical (dumaloq) klasterlar")
print("  - Yaxshi ajratilgan klasterlar")
print("  - O'xshash o'lchamdagi klasterlar")
print("\n❌ Yomon ishlaydi:")
print("  - Non-spherical (moons, ellipse) shakl")
print("  - Nested (ichma-ich) klasterlar")
print("  - Turli o'lchamdagi klasterlar")
print("  - Ko'p outlier'lar")
print("="*70)

---

# 2️⃣ Hierarchical Clustering

## 📖 Nazariya

**Hierarchical Clustering** - klasterlarni **daraxtsimon tarzda** quradigan algoritm.

### Afzalliklari:
- **K tanlash shart emas** - dendrogram'dan ko'rib tanlaymiz
- **Hierarchical structure** - kichik klasterlarni katta klasterlarga birlashtiradi
- **Visualization** - dendrogram orqali ko'rinadi

### 2 xil Hierarchical Clustering:

#### 1. **Agglomerative (Bottom-up)** ⬆️
- Har bir nuqta alohida klaster
- Step-by-step eng yaqin klasterlarni birlashtirish
- Oxirida bitta katta klaster

#### 2. **Divisive (Top-down)** ⬇️
- Barcha nuqta bitta klasterda
- Step-by-step klasterni ajratish
- Oxirida har bir nuqta alohida

**Biz Agglomerative'dan foydalanamiz** (sklearn'da ham shu bor).

---

## Linkage Methods

Ikki klaster orasidagi masofani qanday hisoblash kerak?

### 1. **Single Linkage** (Minimum)
$$d(C_1, C_2) = \min_{x \in C_1, y \in C_2} ||x - y||$$

Eng yaqin nuqtalar orasidagi masofa.

### 2. **Complete Linkage** (Maximum)
$$d(C_1, C_2) = \max_{x \in C_1, y \in C_2} ||x - y||$$

Eng uzoq nuqtalar orasidagi masofa.

### 3. **Average Linkage**
$$d(C_1, C_2) = \frac{1}{|C_1| \cdot |C_2|} \sum_{x \in C_1} \sum_{y \in C_2} ||x - y||$$

Barcha nuqtalar orasidagi o'rtacha masofa.

### 4. **Ward Linkage** (Eng mashhur)
$$d(C_1, C_2) = \text{Inertia increase agar birlashtirilsa}$$

Inertia'ni minimal oshiradigan klasterlarni birlashtiradi.

---

## 🎨 Hierarchical Clustering: Vizual Demo

In [ ]:
# Oddiy 2D data yaratish
np.random.seed(42)
X_hier, _ = make_blobs(n_samples=100, centers=3, cluster_std=0.8, random_state=42)

# Linkage matrix hisoblash (Ward method)
linkage_matrix = linkage(X_hier, method='ward')

# Dendrogram chizish
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Dendrogram
dendrogram(linkage_matrix, ax=axes[0], color_threshold=10)
axes[0].set_title('Dendrogram (Ward Linkage)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sample Index', fontsize=12)
axes[0].set_ylabel('Distance', fontsize=12)
axes[0].axhline(y=10, color='red', linestyle='--', linewidth=2, label='Cut threshold=10')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# Hierarchical Clustering (K=3)
hier_clust = AgglomerativeClustering(n_clusters=3, linkage='ward')
hier_labels = hier_clust.fit_predict(X_hier)

axes[1].scatter(X_hier[:, 0], X_hier[:, 1], c=hier_labels, 
                s=100, alpha=0.7, cmap='viridis', edgecolors='k', linewidth=1)
axes[1].set_title('Hierarchical Clustering (K=3)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Feature 1', fontsize=12)
axes[1].set_ylabel('Feature 2', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("DENDROGRAM TUSHUNTIRISH")
print("="*70)
print("\n📊 Dendrogram nima?")
print("  - Daraxtsimon diagram: klasterlar qanday birlashishini ko'rsatadi")
print("  - Y o'qi: klasterlar orasidagi masofa")
print("  - X o'qi: har bir sample (nuqta)")
print("\n💡 Qanday o'qiladi?")
print("  - Pastdan tepaga: kichik klasterlar katta klasterlarga birlashadi")
print("  - Gorizontal chiziq (qizil): 'cut' - nechta klaster kerakligini belgilaydi")
print("  - Qizil chiziq ostida 3 ta branch: 3 ta klaster")
print("\n🎯 K ni tanlash:")
print("  - Uzun vertikal chiziq: yaxshi ajratilgan klasterlar")
print("  - O'sha joyda 'cut' qilish kerak!")
print("="*70)

## 🔍 Linkage Methods Comparison

Keling, turli linkage metodlarini taqqoslaylik:

In [ ]:
# Linkage methods comparison
linkage_methods = ['single', 'complete', 'average', 'ward']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx, method in enumerate(linkage_methods):
    # Dendrogram
    linkage_mat = linkage(X_hier, method=method)
    dendrogram(linkage_mat, ax=axes[0, idx], no_labels=True)
    axes[0, idx].set_title(f'{method.capitalize()} Linkage\nDendrogram', 
                           fontsize=12, fontweight='bold')
    axes[0, idx].set_ylabel('Distance', fontsize=10)
    axes[0, idx].grid(True, alpha=0.3, axis='y')
    
    # Clustering result
    hier_method = AgglomerativeClustering(n_clusters=3, linkage=method)
    labels_method = hier_method.fit_predict(X_hier)
    
    axes[1, idx].scatter(X_hier[:, 0], X_hier[:, 1], c=labels_method,
                         s=80, alpha=0.7, cmap='viridis', edgecolors='k', linewidth=1)
    axes[1, idx].set_title(f'{method.capitalize()} Linkage\nClustering', 
                           fontsize=12, fontweight='bold')
    axes[1, idx].set_xlabel('Feature 1', fontsize=10)
    axes[1, idx].set_ylabel('Feature 2', fontsize=10)
    axes[1, idx].grid(True, alpha=0.3)

plt.suptitle('Linkage Methods Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("LINKAGE METHODS TAQQOSLASH")
print("="*70)
print("\n🔗 Single Linkage:")
print("  - Eng yaqin nuqtalar orasidagi masofa")
print("  - Afzallik: Har xil shakldagi klasterlar")
print("  - Kamchilik: 'Chaining' - uzun zanjir hosil qiladi")
print("\n🔗 Complete Linkage:")
print("  - Eng uzoq nuqtalar orasidagi masofa")
print("  - Afzallik: Compact (zich) klasterlar")
print("  - Kamchilik: Outlier'larga sezgir")
print("\n🔗 Average Linkage:")
print("  - O'rtacha masofa")
print("  - Afzallik: Balans - Single va Complete o'rtasida")
print("  - Kamchilik: Hisoblash qimmat")
print("\n🔗 Ward Linkage: ⭐ (Eng mashhur)")
print("  - Inertia'ni minimal oshiradi")
print("  - Afzallik: Balansli, yaxshi klasterlar")
print("  - Kamchilik: Faqat Euclidean distance")
print("="*70)

## 💻 Hierarchical Clustering: Amaliy Misol (Wine Dataset)

In [ ]:
# Wine datasetini yuklash
wine = load_wine()
X_wine = wine.data
y_wine_true = wine.target  # Faqat compare uchun

# Standardization (important for hierarchical clustering!)
scaler = StandardScaler()
X_wine_scaled = scaler.fit_transform(X_wine)

# PCA: 2D visualization uchun (tushuntirish keyinroq)
pca_wine = PCA(n_components=2)
X_wine_2d = pca_wine.fit_transform(X_wine_scaled)

# Hierarchical Clustering
hier_wine = AgglomerativeClustering(n_clusters=3, linkage='ward')
y_pred_wine = hier_wine.fit_predict(X_wine_scaled)

# Linkage matrix (dendrogram uchun)
linkage_wine = linkage(X_wine_scaled, method='ward')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Dendrogram
dendrogram(linkage_wine, ax=axes[0], color_threshold=50, no_labels=True)
axes[0].set_title('Wine Dataset - Dendrogram', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sample Index', fontsize=12)
axes[0].set_ylabel('Distance', fontsize=12)
axes[0].axhline(y=50, color='red', linestyle='--', linewidth=2, label='Cut threshold=50')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# Clustering result (2D PCA)
axes[1].scatter(X_wine_2d[:, 0], X_wine_2d[:, 1], c=y_pred_wine,
                s=100, alpha=0.7, cmap='viridis', edgecolors='k', linewidth=1)
axes[1].set_title('Wine Dataset - Hierarchical Clustering (3 clusters)', 
                  fontsize=14, fontweight='bold')
axes[1].set_xlabel('PC1 (Principal Component 1)', fontsize=12)
axes[1].set_ylabel('PC2 (Principal Component 2)', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Metrics
silhouette_wine = silhouette_score(X_wine_scaled, y_pred_wine)

print("\n" + "="*60)
print("HIERARCHICAL CLUSTERING NATIJALAR (Wine Dataset)")
print("="*60)
print(f"Silhouette Score: {silhouette_wine:.4f}")
print(f"\n📊 Cluster sizes:")
unique_wine, counts_wine = np.unique(y_pred_wine, return_counts=True)
for cluster, count in zip(unique_wine, counts_wine):
    print(f"  Cluster {cluster}: {count} samples")
print("\n💡 PCA: 13D → 2D (visualization uchun)")
print(f"   Explained variance: {pca_wine.explained_variance_ratio_.sum():.2%}")
print("="*60)

---

# 3️⃣ PCA (Principal Component Analysis)

## 📖 Nazariya

**PCA** - **Dimensionality Reduction** algoritmi. Ko'p o'lchamli datani kam o'lchamga aylantiradi.

### Nima uchun kerak?

1. **Visualization** - 10D datani 2D/3D'da ko'rish
2. **Storage** - Kam joy egallaydi
3. **Speed** - Tez ishlaydi
4. **Noise reduction** - Shovqinni olib tashlaydi
5. **Feature engineering** - Yangi feature'lar yaratish

### Qanday ishlaydi?

PCA yangi koordinata sistemasi yaratadi:
- **Principal Component 1 (PC1)**: Eng ko'p variance yo'nalishi
- **Principal Component 2 (PC2)**: 2-eng ko'p variance (PC1'ga perpendikulyar)
- **PC3, PC4, ...**: Navbatdagi yo'nalishlar

### Matematik Tushuntirish

**1. Standardization** (muhim!):
$$X_{scaled} = \frac{X - \mu}{\sigma}$$

**2. Covariance Matrix**:
$$\Sigma = \frac{1}{n-1} X^T X$$

**3. Eigenvalue & Eigenvector**:
$$\Sigma v = \lambda v$$

Bu yerda:
- $v$ - Eigenvector (yangi yo'nalish, PC)
- $\lambda$ - Eigenvalue (variance)

**4. Projection** (transform):
$$X_{new} = X \cdot V$$

$V$ - Eigenvector'lar matritsasi (PC'lar)

---

## 🎯 Explained Variance

Har bir PC qancha variance tushuntiradi?

**Explained Variance Ratio**:
$$\text{Explained Variance Ratio}_i = \frac{\lambda_i}{\sum_{j=1}^{p} \lambda_j}$$

Misol:
- PC1: 50% variance
- PC2: 30% variance
- PC3: 15% variance
- PC4: 5% variance

**Cumulative Variance**: 50% + 30% = 80% (PC1+PC2)

💡 **Rule of thumb**: 80-95% variance saqlanadi.

---

## 🎨 PCA: Vizual Tushuntirish (2D → 1D)

In [ ]:
# 2D data yaratish (correlated features)
np.random.seed(42)
mean = [0, 0]
cov = [[3, 2.5], [2.5, 3]]  # Covariance matrix (correlation mavjud)
X_pca_demo = np.random.multivariate_normal(mean, cov, 200)

# PCA (2D → 1D)
pca_demo = PCA(n_components=2)
X_pca_transformed = pca_demo.fit_transform(X_pca_demo)

# Principal Components (eigenvectors)
pc1 = pca_demo.components_[0]
pc2 = pca_demo.components_[1]

# Explained variance
exp_var = pca_demo.explained_variance_ratio_

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Original Data + PC arrows
axes[0].scatter(X_pca_demo[:, 0], X_pca_demo[:, 1], s=50, alpha=0.6, 
                c='blue', edgecolors='k')
axes[0].arrow(0, 0, pc1[0]*3, pc1[1]*3, head_width=0.3, head_length=0.3, 
              fc='red', ec='red', linewidth=3, label=f'PC1 ({exp_var[0]:.1%} var)')
axes[0].arrow(0, 0, pc2[0]*3, pc2[1]*3, head_width=0.3, head_length=0.3,
              fc='green', ec='green', linewidth=3, label=f'PC2 ({exp_var[1]:.1%} var)')
axes[0].set_xlabel('Feature 1', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Feature 2', fontsize=12, fontweight='bold')
axes[0].set_title('Original Data + Principal Components', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

# Transformed Data (2D - PC1 and PC2)
axes[1].scatter(X_pca_transformed[:, 0], X_pca_transformed[:, 1], s=50, 
                alpha=0.6, c='purple', edgecolors='k')
axes[1].axhline(y=0, color='red', linewidth=2, label='PC1 axis')
axes[1].axvline(x=0, color='green', linewidth=2, label='PC2 axis')
axes[1].set_xlabel('PC1', fontsize=12, fontweight='bold')
axes[1].set_ylabel('PC2', fontsize=12, fontweight='bold')
axes[1].set_title('Transformed Data (New Coordinate System)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].axis('equal')

# 1D projection (faqat PC1)
axes[2].scatter(X_pca_transformed[:, 0], np.zeros_like(X_pca_transformed[:, 0]),
                s=50, alpha=0.6, c='orange', edgecolors='k')
axes[2].axhline(y=0, color='red', linewidth=2)
axes[2].set_xlabel('PC1', fontsize=12, fontweight='bold')
axes[2].set_ylabel('(Removed)', fontsize=12, fontweight='bold')
axes[2].set_title(f'1D Projection (PC1 faqat, {exp_var[0]:.1%} variance)', 
                  fontsize=14, fontweight='bold')
axes[2].set_ylim(-1, 1)
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("PCA VIZUAL TUSHUNTIRISH")
print("="*70)
print("\n📊 Chap grafik: Original 2D data")
print("  - Qizil strelka: PC1 (maksimal variance yo'nalishi)")
print("  - Yashil strelka: PC2 (PC1'ga perpendikulyar)")
print(f"  - PC1: {exp_var[0]:.1%} variance")
print(f"  - PC2: {exp_var[1]:.1%} variance")
print("\n📊 O'rta grafik: Transformed data (yangi koordinata)")
print("  - X o'qi: PC1")
print("  - Y o'qi: PC2")
print("  - Data aylantirildi, lekin shape bir xil")
print("\n📊 O'ng grafik: 1D projection")
print("  - Faqat PC1 qoldirildi, PC2 olib tashlandi")
print(f"  - {exp_var[0]:.1%} variance saqlanadi")
print(f"  - Dimensionality: 2D → 1D")
print("="*70)

## 💻 PCA: Amaliy Misol (Iris Dataset)

In [ ]:
# Iris dataset: 4D → 2D
# Iris data allaqachon yuklangan: X_iris, y_iris_true

# Standardization
scaler_iris = StandardScaler()
X_iris_scaled = scaler_iris.fit_transform(X_iris)

# PCA: 4D → 2D
pca_iris = PCA(n_components=2)
X_iris_pca = pca_iris.fit_transform(X_iris_scaled)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Original data (faqat 2 feature ko'rsatish mumkin)
axes[0].scatter(X_iris_scaled[:, 0], X_iris_scaled[:, 1], c=y_iris_true,
                s=100, alpha=0.7, cmap='viridis', edgecolors='k', linewidth=1)
axes[0].set_xlabel('Feature 1 (Sepal Length)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Feature 2 (Sepal Width)', fontsize=12, fontweight='bold')
axes[0].set_title('Original Data (faqat 2 feature)', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# PCA transformed (PC1 vs PC2)
scatter = axes[1].scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], c=y_iris_true,
                          s=100, alpha=0.7, cmap='viridis', edgecolors='k', linewidth=1)
axes[1].set_xlabel(f'PC1 ({pca_iris.explained_variance_ratio_[0]:.1%} variance)', 
                   fontsize=12, fontweight='bold')
axes[1].set_ylabel(f'PC2 ({pca_iris.explained_variance_ratio_[1]:.1%} variance)', 
                   fontsize=12, fontweight='bold')
axes[1].set_title('PCA Transformed (4D → 2D)', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Colorbar
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('Species', fontsize=11)

plt.tight_layout()
plt.show()

# Explained Variance
print("\n" + "="*60)
print("PCA NATIJALAR (Iris Dataset)")
print("="*60)
print(f"\nOriginal dimensions: {X_iris.shape[1]}D")
print(f"Reduced dimensions: {pca_iris.n_components_}D")
print(f"\nExplained Variance Ratio:")
for i, var in enumerate(pca_iris.explained_variance_ratio_):
    print(f"  PC{i+1}: {var:.2%}")
print(f"\nCumulative Variance: {pca_iris.explained_variance_ratio_.sum():.2%}")
print(f"\n💡 {pca_iris.explained_variance_ratio_.sum():.1%} variance saqlanadi!")
print("="*60)

## 📈 Scree Plot: Nechta PC kerak?

**Scree Plot** - Har bir PC'ning explained variance'ini ko'rsatadi.

In [ ]:
# PCA: Barcha PC'lar
pca_all = PCA()
pca_all.fit(X_iris_scaled)

# Explained variance
exp_var_all = pca_all.explained_variance_ratio_
cum_var_all = np.cumsum(exp_var_all)

# Scree Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Individual explained variance
axes[0].bar(range(1, len(exp_var_all)+1), exp_var_all, alpha=0.7, color='steelblue', 
            edgecolor='black', linewidth=1.5)
axes[0].plot(range(1, len(exp_var_all)+1), exp_var_all, 'ro-', linewidth=2, markersize=10)
axes[0].set_xlabel('Principal Component', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Explained Variance Ratio', fontsize=12, fontweight='bold')
axes[0].set_title('Scree Plot - Individual Variance', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(1, len(exp_var_all)+1))
axes[0].grid(True, alpha=0.3, axis='y')

# Cumulative explained variance
axes[1].plot(range(1, len(cum_var_all)+1), cum_var_all, 'bo-', linewidth=2, markersize=10)
axes[1].axhline(y=0.95, color='red', linestyle='--', linewidth=2, label='95% threshold')
axes[1].axhline(y=0.80, color='orange', linestyle='--', linewidth=2, label='80% threshold')
axes[1].set_xlabel('Number of Components', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Cumulative Explained Variance', fontsize=12, fontweight='bold')
axes[1].set_title('Cumulative Variance', fontsize=14, fontweight='bold')
axes[1].set_xticks(range(1, len(cum_var_all)+1))
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("SCREE PLOT TUSHUNTIRISH")
print("="*70)
print("\n📊 Individual Variance (chap):")
print("  - Har bir PC qancha variance tushuntiradi")
print("  - Birinchi PC'lar ko'proq variance")
print("\n📊 Cumulative Variance (o'ng):")
print("  - Jami qancha variance saqlanadi")
print("  - 95% yoki 80% threshold'ga yetganda to'xtash mumkin")
print("\n💡 Ushbu misolda:")
for i, (ind_var, cum_var) in enumerate(zip(exp_var_all, cum_var_all)):
    print(f"  {i+1} PC: {ind_var:.2%} (cumulative: {cum_var:.2%})")
print(f"\n🎯 2 PC yetarli: {cum_var_all[1]:.1%} variance saqlanadi!")
print("="*70)

---

# 🔄 Algorithms Comparison

## K-means vs Hierarchical vs PCA

| **Xususiyat** | **K-means** | **Hierarchical** | **PCA** |
|---------------|-------------|------------------|---------|
| **Turi** | Clustering | Clustering | Dimensionality Reduction |
| **K tanlash** | Kerak ✅ | Kerak emas ❌ | n_components |
| **Hisoblash** | Tez ⚡ | Sekin 🐌 | O'rtacha ⏱️ |
| **Scalability** | Katta data ✅ | Kichik data ⚠️ | Katta data ✅ |
| **Visualization** | Scatter plot | Dendrogram | PC1 vs PC2 |
| **Outliers** | Sezgir ⚠️ | Kam sezgir ✅ | Kam sezgir ✅ |
| **Shape** | Spherical faqat ⚠️ | Har xil ✅ | - |
| **Interpretability** | O'rtacha | Yaxshi ✅ | Qiyin ⚠️ |

---

## 🎬 Real-World Application: Customer Segmentation

Keling, haqiqiy misolda barcha algoritmlarni birga ishlatamiz!

**Vazifa**: Mijozlarni guruhlash (segmentation)

In [ ]:
# Synthetic customer data
np.random.seed(42)
n_customers = 500

# Features: Age, Income, Spending Score, Purchase Frequency
age = np.random.normal(40, 15, n_customers)
income = np.random.normal(60000, 20000, n_customers)
spending = np.random.normal(50, 25, n_customers)
frequency = np.random.normal(10, 5, n_customers)

X_customers = np.column_stack([age, income, spending, frequency])
X_customers = np.clip(X_customers, 0, None)  # No negative values

# Standardization
scaler_cust = StandardScaler()
X_customers_scaled = scaler_cust.fit_transform(X_customers)

# Step 1: PCA (4D → 2D for visualization)
pca_cust = PCA(n_components=2)
X_customers_pca = pca_cust.fit_transform(X_customers_scaled)

# Step 2: Elbow Method (optimal K)
inertias_cust = []
K_range_cust = range(2, 11)
for k in K_range_cust:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_temp.fit(X_customers_scaled)
    inertias_cust.append(kmeans_temp.inertia_)

# Step 3: K-means clustering (K=4)
kmeans_cust = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_kmeans = kmeans_cust.fit_predict(X_customers_scaled)

# Step 4: Hierarchical clustering (K=4)
hier_cust = AgglomerativeClustering(n_clusters=4, linkage='ward')
labels_hier = hier_cust.fit_predict(X_customers_scaled)

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(20, 13))

# Elbow Method
axes[0, 0].plot(K_range_cust, inertias_cust, 'bo-', linewidth=2, markersize=10)
axes[0, 0].set_xlabel('K (Number of Clusters)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Inertia (WCSS)', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Elbow Method', fontsize=13, fontweight='bold')
axes[0, 0].axvline(x=4, color='red', linestyle='--', linewidth=2, label='Optimal K=4')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# K-means result
scatter1 = axes[0, 1].scatter(X_customers_pca[:, 0], X_customers_pca[:, 1], 
                              c=labels_kmeans, s=60, alpha=0.7, cmap='viridis', 
                              edgecolors='k', linewidth=0.5)
axes[0, 1].set_xlabel('PC1', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('PC2', fontsize=11, fontweight='bold')
axes[0, 1].set_title('K-means Clustering (K=4)', fontsize=13, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=axes[0, 1], label='Cluster')

# Hierarchical result
scatter2 = axes[0, 2].scatter(X_customers_pca[:, 0], X_customers_pca[:, 1],
                              c=labels_hier, s=60, alpha=0.7, cmap='viridis',
                              edgecolors='k', linewidth=0.5)
axes[0, 2].set_xlabel('PC1', fontsize=11, fontweight='bold')
axes[0, 2].set_ylabel('PC2', fontsize=11, fontweight='bold')
axes[0, 2].set_title('Hierarchical Clustering (K=4)', fontsize=13, fontweight='bold')
axes[0, 2].grid(True, alpha=0.3)
plt.colorbar(scatter2, ax=axes[0, 2], label='Cluster')

# Cluster Statistics - K-means
df_stats = pd.DataFrame(X_customers, columns=['Age', 'Income', 'Spending', 'Frequency'])
df_stats['Cluster'] = labels_kmeans
cluster_means = df_stats.groupby('Cluster').mean()

axes[1, 0].axis('off')
table_data = []
for i in range(4):
    table_data.append([
        f"Cluster {i}",
        f"{cluster_means.loc[i, 'Age']:.1f}",
        f"${cluster_means.loc[i, 'Income']:.0f}",
        f"{cluster_means.loc[i, 'Spending']:.1f}",
        f"{cluster_means.loc[i, 'Frequency']:.1f}"
    ])

table = axes[1, 0].table(cellText=table_data,
                         colLabels=['Cluster', 'Avg Age', 'Avg Income', 'Avg Spending', 'Avg Freq'],
                         cellLoc='center', loc='center',
                         colWidths=[0.15, 0.15, 0.2, 0.25, 0.25])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)
for i in range(5):
    table[(0, i)].set_facecolor('#40466e')
    table[(0, i)].set_text_props(weight='bold', color='white')
axes[1, 0].set_title('Cluster Statistics (K-means)', fontsize=13, fontweight='bold', pad=20)

# Feature distributions by cluster
for idx, feature in enumerate(['Age', 'Income']):
    ax = axes[1, idx+1]
    for cluster in range(4):
        cluster_data = df_stats[df_stats['Cluster'] == cluster][feature]
        ax.hist(cluster_data, bins=20, alpha=0.6, label=f'Cluster {cluster}', edgecolor='black')
    ax.set_xlabel(feature, fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax.set_title(f'{feature} Distribution by Cluster', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Customer Segmentation: Complete Pipeline', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Metrics
sil_kmeans = silhouette_score(X_customers_scaled, labels_kmeans)
sil_hier = silhouette_score(X_customers_scaled, labels_hier)

print("\n" + "="*70)
print("CUSTOMER SEGMENTATION NATIJALAR")
print("="*70)
print(f"\n📊 Dataset: {n_customers} customers, 4 features")
print(f"   Features: Age, Income, Spending Score, Purchase Frequency")
print(f"\n🔍 PCA: 4D → 2D")
print(f"   Explained variance: {pca_cust.explained_variance_ratio_.sum():.2%}")
print(f"\n🎯 Clustering Results:")
print(f"   K-means Silhouette Score: {sil_kmeans:.4f}")
print(f"   Hierarchical Silhouette Score: {sil_hier:.4f}")
print(f"\n💡 Cluster Interpretation:")
for i in range(4):
    count = (labels_kmeans == i).sum()
    print(f"   Cluster {i}: {count} customers")
    print(f"      Age: {cluster_means.loc[i, 'Age']:.1f}, Income: ${cluster_means.loc[i, 'Income']:.0f}")
    print(f"      Spending: {cluster_means.loc[i, 'Spending']:.1f}, Frequency: {cluster_means.loc[i, 'Frequency']:.1f}")
print("="*70)

---

# 📝 Xulosa (Summary)

## Unsupervised Learning: Asosiy Fikrlar

### 1️⃣ K-means Clustering
- **Vazifa**: Ma'lumotlarni K ta klasterga ajratish
- **Qachon ishlatish**: 
  - Spherical (dumaloq) klasterlar
  - Tez natija kerak
  - Katta dataset
- **Optimal K**: Elbow Method, Silhouette Score
- **Afzallik**: Tez, oddiy
- **Kamchilik**: K tanlash qiyin, outlier'larga sezgir, faqat spherical

### 2️⃣ Hierarchical Clustering
- **Vazifa**: Daraxtsimon klasterlash
- **Qachon ishlatish**:
  - K noma'lum
  - Hierarchical structure muhim
  - Kichik dataset
- **Linkage**: Ward (eng yaxshi), Single, Complete, Average
- **Afzallik**: K tanlash shart emas, har xil shakl
- **Kamchilik**: Sekin, katta data bilan qiyin

### 3️⃣ PCA (Principal Component Analysis)
- **Vazifa**: Dimensionality reduction (n-D → k-D)
- **Qachon ishlatish**:
  - Visualization (2D/3D)
  - Ko'p feature'lar (curse of dimensionality)
  - Correlated features
  - Noise reduction
- **Nechta PC**: Scree plot, 80-95% variance
- **Afzallik**: Tez, storage kam, visualization
- **Kamchilik**: Interpretability qiyin, linear faqat

---

## 🎯 Best Practices

### Data Preprocessing
1. **Standardization** - DOIM ishlatish! (K-means, Hierarchical, PCA)
2. **Outliers** - olib tashlash yoki alohida ko'rish
3. **Missing values** - to'ldirish yoki olib tashlash

### K-means
1. **n_init** ni katta qilish (10+) - random initialization
2. **Elbow Method + Silhouette** - K ni tanlash
3. **Scaling** - DOIM standardize qilish

### Hierarchical
1. **Ward linkage** - odatda eng yaxshi
2. **Dendrogram** - K ni vizual tanlash
3. **Kichik dataset** - katta data bilan sekin

### PCA
1. **Standardization** - DOIM!
2. **Explained variance** - 80-95% saqlanishi kerak
3. **Scree plot** - nechta PC kerakligini ko'rish
4. **Interpretation** - PC'larni tushunish qiyin

---

## 🚀 Keyingi Qadamlar

### Amaliyot uchun:
1. **practical.ipynb** - amaliy mashqlar
2. **homework.md** - uyga vazifa
3. **unsupervised_guide.md** - tez qo'llanma

### O'rganish uchun:
- DBSCAN clustering (density-based)
- t-SNE (non-linear dimensionality reduction)
- Autoencoders (neural network-based)
- Anomaly detection

---

# ✅ Dars tugadi!

**Nima o'rgandik:**
- ✅ K-means: Clustering algoritmi
- ✅ Hierarchical: Daraxtsimon klasterlash
- ✅ PCA: Dimensionality reduction
- ✅ Real-world application: Customer segmentation

**Keyingi dars**: Qo'shimcha unsupervised learning algoritmlari

---

## 📚 Qo'shimcha Resurslar

- [Scikit-learn Clustering](https://scikit-learn.org/stable/modules/clustering.html)
- [PCA Explained](https://builtin.com/data-science/step-step-explanation-principal-component-analysis)
- [Hierarchical Clustering Tutorial](https://stackabuse.com/hierarchical-clustering-with-python-and-scikit-learn/)

**Savol bo'lsa - so'rang! 🙋‍♂️**